# F4 — Entrenamiento TimeGAN en Kaggle (T4 GPU)

**Escape hatch GPU**: ejecuta `scripts/03_train_timegan.py` en Kaggle si quieres bajar de ~3-6h (M4 Pro MPS) a ~1-2h (T4).

Flow:
1. Setup: clona repo + instala deps + verifica GPU.
2. Inputs: ejecuta `scripts/02_build_features.py` (self-contained ~5 min).
3. Smoke test (~5 min en T4).
4. Multirun 3 seeds (~1-2h).
5. Evaluación gate F4-T14.
6. Empaqueta artefactos en `.tar.gz` para descarga al Mac.

**Tras descarga local**: desempaca y corre `scripts/04_evaluate_synthetic.py` en el Mac para regenerar el summary.

## 1. Setup: clonar repo + dependencias

**TODO antes de ejecutar**: reemplaza `<user>` y `<repo>` por tu fork público en GitHub (Kaggle no tiene acceso a repos privados sin token).
Si tu repo es privado, usa la opción de "Add Data" para subir un dataset Kaggle pre-creado con `data/processed/` + `models/scalers/` + `data/splits/`.

In [ ]:
!git clone https://github.com/<user>/<repo>.git /kaggle/working/model-tfm
%cd /kaggle/working/model-tfm
!pip install -q \
    'torch>=2.2,<2.5' \
    'hydra-core>=1.3,<2.0' \
    'omegaconf>=2.3,<3.0' \
    'mlflow>=2.10,<3.0' \
    'ta>=0.11,<0.12' \
    'scikit-learn>=1.4,<2.0' \
    'joblib>=1.3,<2.0' \
    'pandas>=2.2,<2.3' \
    'numpy>=1.26,<2.0' \
    'pyarrow>=15.0,<18.0' \
    'yfinance>=0.2.40,<0.3' \
    'pandas-datareader>=0.10.0,<0.11' \
    'pandas_market_calendars>=4.4,<5.0' \
    'statsmodels>=0.14,<0.15' \
    'matplotlib>=3.8,<4.0'

In [ ]:
# Verifica GPU: debe imprimir 'Tesla T4' o similar.
import torch
assert torch.cuda.is_available(), 'No GPU detectada — ve a Settings → Accelerator → GPU T4'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

## 2. Generar datos de entrada

Reproducimos las Fases 1-3 en Kaggle (descarga yfinance + features + scalers + secuencias). Tarda ~5-7 min.

In [ ]:
!python scripts/01_download_data.py
!python scripts/02_build_features.py

In [ ]:
# Verificar que los inputs están listos
import numpy as np
seqs = np.load('data/processed/timegan_train_sequences.npy', mmap_mode='r')
print('Sequences shape:', seqs.shape, 'dtype:', seqs.dtype)
print('Range:', seqs.min(), seqs.max())

## 3. Smoke test (~5 min en T4)

Validar que el pipeline completo ejecuta sin crash antes de comprometer 1-2h.

In [ ]:
!python scripts/03_train_timegan.py timegan/training=smoke seed=42 timegan.device=cuda
!python scripts/04_evaluate_synthetic.py seed=42 timegan.device=cuda

## 4. Multirun full (3 seeds, ~1-2h en T4)

Si el smoke test pasa, lanzar el multirun completo. Kaggle T4 tiene timeout de 12h por sesión así que sobra margen.

In [ ]:
# Forzar recálculo (smoke ya dejó artefactos para seed=42)
!python scripts/03_train_timegan.py --multirun seed=0,42,123 timegan.device=cuda force=true

In [ ]:
# Gate de calidad
!python scripts/04_evaluate_synthetic.py --multirun seed=0,42,123 timegan.device=cuda

In [ ]:
# Comprobar flags
import os
for seed in (0, 42, 123):
    flag = f'data/synthetic/run_{seed}/QUALITY_OK.flag'
    status = 'PASS' if os.path.exists(flag) else 'FAIL'
    print(f'seed={seed}: {status}')

## 5. Empaquetado para descarga al Mac

El `.tar.gz` aparecerá en el panel "Output" del notebook (~150-250 MB). Tras descargar:

```bash
tar xzf ~/Downloads/f4_artifacts.tar.gz -C ~/Documents/Universidad/Q2/TFM/model-tfm/
cd ~/Documents/Universidad/Q2/TFM/model-tfm
uv run python scripts/04_evaluate_synthetic.py --multirun seed=0,42,123
```

La segunda línea regenera el `outputs/timegan/timegan_summary.md` con los thresholds del config local.

In [ ]:
!tar czf /kaggle/working/f4_artifacts.tar.gz \
    data/synthetic/ \
    models/timegan/ \
    outputs/timegan/ \
    mlruns/
!ls -lh /kaggle/working/f4_artifacts.tar.gz